#Downloading the data and EDA

In [ ]:
#download the data
!wget "https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip"

import zipfile;

# Unzip the downloaded file
zip_ref = zipfile.ZipFile("nlp_getting_started.zip", "r")
zip_ref.extractall()
zip_ref.close()

--2025-06-19 12:02:43--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 192.178.163.207, 74.125.20.207, 108.177.98.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.178.163.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.008s  

2025-06-19 12:02:43 (74.9 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [ ]:
import pandas as pd
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
train_df_shuffled = train_df.sample(frac=1,random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [ ]:
train_df.target.value_counts()

,count
target,
0,4342
1,3271


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [ ]:
import random
random_index = random.randint(0,len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("---\n")

Target: 1 (real disaster)
Text:
Going back to Gainesville will be the death of me

---

Target: 1 (real disaster)
Text:
@HfxStanfield @beelieveDC @DiscoveryCntr what is happening we hear there is runway lighting damage by a contractor.

---

Target: 0 (not real disaster)
Text:
Dc I love you but please obliterate power girl

---

Target: 0 (not real disaster)
Text:
No snowflake in an avalanche ever feels responsible.

---

Target: 1 (real disaster)
Text:
RT @GreenHarvard: Documenting climate change's first major casualty http://t.co/4q4zd7oU34 via @GreenHarvard

---



In [ ]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)

In [ ]:
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

#Converting text to numbers

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None
                                    )

In [ ]:
  #finding the average number of tokens in training dataset
  round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [ ]:
max_vocab_length = 10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [ ]:
#fittig the text vectorizer to training text
text_vectorizer.adapt(train_sentences)

In [ ]:
sample = "Tsunami hit the city"
text_vectorizer([sample])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[525, 244,   2, 182,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [ ]:
random_sentence = random.choice(train_sentences)
print(f"{random_sentence}")
text_vectorizer([random_sentence])

Ginga thinks he can defeat me? Not with my L-Drago Destroy he can't!


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1, 3334,   56,   71,    1,   31,   34,   14,   13,    1,  305,
          56,   98,    0,    0]])>

In [ ]:
words_in_vocab=text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
last_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {last_5_words}")

Number of words in vocab: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


#Creating an embedding layer

In [ ]:
tf.random.set_seed(42)
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim=128,
                             embeddings_initializer="uniform",
                             input_length=max_length,
                             name="embedding_1")
embedding

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding_1, built=False>

In [ ]:
random_sentence = random.choice(train_sentences)
print(f"{random_sentence}")
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

@Welles_7 he was injured. He is a pro bowl back.


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.01525049,  0.03527932,  0.0341214 , ...,  0.04337617,
          0.03329099,  0.0485402 ],
        [ 0.04378667,  0.04808972, -0.03711312, ...,  0.03293538,
         -0.01261649, -0.00293734],
        [ 0.02263428,  0.04804765,  0.04545609, ...,  0.03725329,
         -0.01247972,  0.02157841],
        ...,
        [ 0.01629188, -0.02258297,  0.01447666, ..., -0.04127959,
         -0.02120738, -0.02723709],
        [ 0.01629188, -0.02258297,  0.01447666, ..., -0.04127959,
         -0.02120738, -0.02723709],
        [ 0.01629188, -0.02258297,  0.01447666, ..., -0.04127959,
         -0.02120738, -0.02723709]]], dtype=float32)>

#Model 0 : Baseline Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

#Create tokenization and modelling Pipeline
model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [ ]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Baseline model accuracy: {baseline_score*100:.2f}%")

Baseline model accuracy: 79.27%


In [ ]:
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:10]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0])

In [ ]:
#Creating an evaluation function
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  model_accuracy = accuracy_score(y_true, y_pred)*100
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy": model_accuracy,
                   "precision": model_precision,
                   "recall": model_recall,
                   "f1": model_f1}
  return model_results

In [ ]:
baseline_results = calculate_results(y_true=val_labels,
                                     y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

In [ ]:
import joblib

joblib.dump(model_0, "/content/drive/MyDrive/models/NLP_disaster_tweet_Baseline_model.pkl")


['/content/drive/MyDrive/models/NLP_disaster_tweet_Baseline_model.pkl']

In [ ]:
import joblib

# Load the model
loaded_baseline_model = joblib.load("/content/drive/MyDrive/models/NLP_disaster_tweet_Baseline_model.pkl")

In [ ]:
loaded_baseline_preds = loaded_baseline_model.predict(val_sentences)
loaded_baseline_results = calculate_results(y_true=val_labels,
                                            y_pred=loaded_baseline_preds)
loaded_baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

#Model 1 : Simple Dense model

In [ ]:
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string") #inputs are one dimentional text
x = text_vectorizer(inputs)#convert input text to numbers
x = embedding(x) #creating embedding of the tokenized text
x = layers.GlobalAveragePooling1D()(x) #to get the output for the statement and not each word
outputs = layers.Dense(1, activation="sigmoid")(x)
model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

#compile the model
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_2            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#creating model checkpoints to save the model while training
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath="/content/drive/MyDrive/models/NLP_dense_model.keras",  # or use .h5
    save_best_only=True,
    monitor="val_loss",  # or "val_accuracy"
    mode="min",          # use "max" if monitoring accuracy
    save_weights_only=False
)

In [ ]:
%load_ext tensorboard
#creating tensorboard callbacks to get the progress of epochs and metrics
import datetime
def create_tensorboard_callback(dir_name, experiment_name):
  log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  tensorboard_callback = tf.keras.callbacks.TensorBoard(
      log_dir=log_dir
  )
  print(f"Saving TensorBoard log files to: {log_dir}")
  return tensorboard_callback

# Create directory to save TensorBoard logs
SAVE_DIR = "/content/drive/MyDrive/models/model_logs"

In [ ]:
history_1 = model_1.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences,val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name="simple_dense_model"),
                                   checkpoint_cb
                                   ])

Saving TensorBoard log files to: /content/drive/MyDrive/models/model_logs/simple_dense_model/20250619-123346
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.6355 - loss: 0.6502 - val_accuracy: 0.7585 - val_loss: 0.5348
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8073 - loss: 0.4671 - val_accuracy: 0.7861 - val_loss: 0.4743
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.8523 - loss: 0.3629 - val_accuracy: 0.7913 - val_loss: 0.4620
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8869 - loss: 0.2967 - val_accuracy: 0.7887 - val_loss: 0.4679
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.9058 - loss: 0.2480 - val_accuracy: 0.7808 - val_loss: 0.4833


In [ ]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7688 - loss: 0.5161 


[0.48333740234375, 0.7808399200439453]

In [ ]:
embedding.weights

[<Variable path=embedding_1/embeddings, shape=(10000, 128), dtype=float32, value=[[ 0.00196181 -0.00736506  0.00326891 ... -0.02587321 -0.02995348
   -0.02723757]
  [-0.05805328 -0.01285781 -0.02529103 ...  0.05372412  0.01728678
   -0.03948761]
  [ 0.00283753  0.03033465 -0.05642128 ... -0.00777306 -0.02833139
    0.01534742]
  ...
  [ 0.0046571   0.03524237  0.00781258 ... -0.00144385 -0.0034646
   -0.03061   ]
  [-0.0690075   0.0825113  -0.01001105 ...  0.04082383 -0.02586306
    0.01254783]
  [-0.09174138  0.09332774 -0.03834829 ...  0.04274334 -0.06719444
    0.11037133]]>]

In [ ]:
model_1_pred = model_1.predict(val_sentences)
model_1_preds = tf.squeeze(tf.round(model_1_pred)) # squeeze removes single dimensions
model_1_preds[:20]

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 1.], dtype=float32)>

In [ ]:
# Calculate model_1 metrics
model_1_results = calculate_results(y_true=val_labels,
                                    y_pred=model_1_preds)
model_1_results

{'accuracy': 78.08398950131233,
 'precision': 0.7841274438015813,
 'recall': 0.7808398950131233,
 'f1': 0.7782630349987989}

In [ ]:
# Is our simple Keras model better than our baseline model?
import numpy as np
np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

array([False, False, False, False])

In [ ]:
# Create a helper function to compare our baseline results to new model results
def compare_baseline_to_new_results(baseline_results, new_model_results):
  for key, value in baseline_results.items():
    print(f"Baseline {key}: {value:.2f}, New {key}: {new_model_results[key]:.2f}, Difference: {new_model_results[key]-value:.2f}")

compare_baseline_to_new_results(baseline_results=loaded_baseline_results,
                                new_model_results=model_1_results)

Baseline accuracy: 79.27, New accuracy: 78.08, Difference: -1.18
Baseline precision: 0.81, New precision: 0.78, Difference: -0.03
Baseline recall: 0.79, New recall: 0.78, Difference: -0.01
Baseline f1: 0.79, New f1: 0.78, Difference: -0.01


In [ ]:
# Load model_1
loaded_model_1 = tf.keras.models.load_model("/content/drive/MyDrive/models/NLP_dense_model.keras")

In [ ]:
# Make predictions with the loaded model
loaded_model_1_preds = tf.squeeze(tf.round(loaded_model_1.predict(val_sentences)))
loaded_model_1_preds[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [ ]:
loaded_model_1_results = calculate_results(y_true=val_labels,
                                            y_pred=loaded_model_1_preds)
loaded_model_1_results

{'accuracy': 79.13385826771653,
 'precision': 0.7997458316766562,
 'recall': 0.7913385826771654,
 'f1': 0.7874035967950923}

In [ ]:
%tensorboard --logdir drive/MyDrive/models/model_logs/simple_dense_model

#Visualizing learned embeddings

In [ ]:
words_in_vocab = text_vectorizer.get_vocabulary()

In [ ]:
embed_weights = model_1.get_layer("embedding_1").get_weights()[0]
print(embed_weights.shape)

(10000, 128)


In [ ]:
import io

out_v = io.open("embedding_vectors.tsv","w",encoding="utf-8")
out_m = io.open("embedding_metadata.tsv","w",encoding="utf-8")

# Write embedding vectors and words to file
for num, word in enumerate(words_in_vocab):
  if num == 0:
     continue # skip padding token
  vec = embed_weights[num]
  out_m.write(word + "\n") # write words to file
  out_v.write("\t".join([str(x) for x in vec]) + "\n") # write corresponding word vector to file
out_v.close()
out_m.close()

# Download files locally to upload to Embedding Projector
try:
  from google.colab import files
except ImportError:
  pass
else:
  files.download("embedding_vectors.tsv")
  files.download("embedding_metadata.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Using RNNS

##Model 2 : LSTM

In [ ]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_2_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding_2")

#Creating LSTM model
inputs = layers.Input(shape=(1,),dtype="string")
x=text_vectorizer(inputs)
x=model_2_embedding(x)
x=layers.LSTM(64)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_2 = tf.keras.Model(inputs, outputs, name = "model_2_LSTM")

model_2.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#creating model checkpoints to save the model while training
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath="/content/drive/MyDrive/models/NLP_LSTM.keras",  # or use .h5
    save_best_only=True,
    monitor="val_loss",  # or "val_accuracy"
    mode="min",          # use "max" if monitoring accuracy
    save_weights_only=False
)

history_2 = model_2.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences,val_labels),
                        callbacks=[create_tensorboard_callback(SAVE_DIR,
                                                               "LSTM"),
                                   checkpoint_cb
                                   ])

Saving TensorBoard log files to: /content/drive/MyDrive/models/model_logs/LSTM/20250619-145856
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.6776 - loss: 0.5788 - val_accuracy: 0.7795 - val_loss: 0.4597
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.8634 - loss: 0.3294 - val_accuracy: 0.7625 - val_loss: 0.5044
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.9141 - loss: 0.2293 - val_accuracy: 0.7559 - val_loss: 0.5672
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9402 - loss: 0.1684 - val_accuracy: 0.7598 - val_loss: 0.5932
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.9583 - loss: 0.1259 - val_accuracy: 0.7677 - val_loss: 0.6706


In [ ]:
model_2_pred = model_2.predict(val_sentences)
model_2_preds = tf.squeeze(tf.round(model_2_pred))
model_2_preds[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [ ]:
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 76.77165354330708,
 'precision': 0.7712255031085189,
 'recall': 0.7677165354330708,
 'f1': 0.7646846187166754}

In [ ]:
compare_baseline_to_new_results(loaded_baseline_results, model_2_results)

Baseline accuracy: 79.27, New accuracy: 76.77, Difference: -2.49
Baseline precision: 0.81, New precision: 0.77, Difference: -0.04
Baseline recall: 0.79, New recall: 0.77, Difference: -0.02
Baseline f1: 0.79, New f1: 0.76, Difference: -0.02


##Model 3 : GRU

In [ ]:
tf.random.set_seed(42)
from tensorflow.keras import layers

model_3_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding_3")

#Building GRU model
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_3_embedding(x)
x = layers.GRU(64)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")

model_3.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
model_3.summary()

Model: "model_3_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,317,313 (5.03 MB)

 Trainable params: 1,317,313 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#creating model checkpoints to save the model while training
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath="/content/drive/MyDrive/models/NLP_GRU.keras",  # or use .h5
    save_best_only=True,
    monitor="val_loss",  # or "val_accuracy"
    mode="min",          # use "max" if monitoring accuracy
    save_weights_only=False
)

history_3 = model_3.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences,val_labels),
                        callbacks=[create_tensorboard_callback(SAVE_DIR,"GRU"),
                                   checkpoint_cb]
)

Saving TensorBoard log files to: /content/drive/MyDrive/models/model_logs/GRU/20250619-161256
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.6481 - loss: 0.6094 - val_accuracy: 0.7769 - val_loss: 0.4585
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.8599 - loss: 0.3411 - val_accuracy: 0.7756 - val_loss: 0.5060
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.9092 - loss: 0.2364 - val_accuracy: 0.7638 - val_loss: 0.5700
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.9408 - loss: 0.1646 - val_accuracy: 0.7743 - val_loss: 0.6054
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.9607 - loss: 0.1291 - val_accuracy: 0.7612 - val_loss: 0.6933


In [ ]:
model_3_pred = model_3.predict(val_sentences)
model_3_preds = tf.squeeze(tf.round(model_3_pred))
model_3_preds[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 0., 1., 1., 1.], dtype=float32)>

In [ ]:
model_3_results = calculate_results(y_true=val_labels,
                                    y_pred=model_3_preds)
model_3_results

{'accuracy': 76.11548556430446,
 'precision': 0.7620839190766789,
 'recall': 0.7611548556430446,
 'f1': 0.7592507437677535}

In [ ]:
compare_baseline_to_new_results(loaded_baseline_results, model_3_results)

Baseline accuracy: 79.27, New accuracy: 76.12, Difference: -3.15
Baseline precision: 0.81, New precision: 0.76, Difference: -0.05
Baseline recall: 0.79, New recall: 0.76, Difference: -0.03
Baseline f1: 0.79, New f1: 0.76, Difference: -0.03


##Model 4 : Bidirectional RNN model

In [ ]:
tf.random.set_seed(42)
from tensorflow.keras import layers

model_4_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding_4")

#Create bidirectional model 4
inputs = layers.Input(shape=(1,),dtype="string")
x = text_vectorizer(inputs)
x = model_4_embedding(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_4 = tf.keras.Model(inputs, outputs, name="model_4_Bidirectional")

model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
model_4.summary()

Model: "model_4_Bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,378,945 (5.26 MB)

 Trainable params: 1,378,945 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#creating model checkpoints to save the model while training
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath="/content/drive/MyDrive/models/NLP_Bidirectional.keras",  # or use .h5
    save_best_only=True,
    monitor="val_loss",  # or "val_accuracy"
    mode="min",          # use "max" if monitoring accuracy
    save_weights_only=False
)

history_4 = model_4.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences,val_labels),
                        callbacks=create_tensorboard_callback(SAVE_DIR,"bidirectional_RNN"))

Saving TensorBoard log files to: /content/drive/MyDrive/models/model_logs/bidirectional_RNN/20250619-165626
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 35s 104ms/step - accuracy: 0.6739 - loss: 0.5850 - val_accuracy: 0.7795 - val_loss: 0.4618
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 37s 87ms/step - accuracy: 0.8651 - loss: 0.3309 - val_accuracy: 0.7677 - val_loss: 0.5039
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.9123 - loss: 0.2266 - val_accuracy: 0.7507 - val_loss: 0.5983
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.9411 - loss: 0.1583 - val_accuracy: 0.7428 - val_loss: 0.7058
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 18s 43ms/step - accuracy: 0.9655 - loss: 0.1158 - val_accuracy: 0.7467 - val_loss: 0.7121


In [ ]:
model_4_pred = model_4.predict(val_sentences)
model_4_preds = tf.squeeze(tf.round(model_4_pred))
model_4_preds[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [ ]:
model_4_results = calculate_results(y_true=val_labels,
                                    y_pred=model_4_preds)
model_4_results

{'accuracy': 74.67191601049869,
 'precision': 0.7465631385096111,
 'recall': 0.7467191601049868,
 'f1': 0.7453858813570736}

In [ ]:
compare_baseline_to_new_results(loaded_baseline_results, model_4_results)

Baseline accuracy: 79.27, New accuracy: 74.67, Difference: -4.59
Baseline precision: 0.81, New precision: 0.75, Difference: -0.06
Baseline recall: 0.79, New recall: 0.75, Difference: -0.05
Baseline f1: 0.79, New f1: 0.75, Difference: -0.04


#Model 5 : Conv1D

In [ ]:
tf.random.set_seed(42)
from tensorflow.keras import layers

model_5_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding_5")

#Create Conv1D model 5
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_5_embedding(x)
x = layers.Conv1D(filters=32, kernel_size=5, activation="relu")(x)
x = layers.GlobalMaxPool1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

model_5.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
model_5.summary()

Model: "model_5_Conv1D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 11, 32)         │        20,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_4          │ (None, 32)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,300,545 (4.96 MB)

 Trainable params: 1,300,545 (4.96 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#creating model checkpoints to save the model while training
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath="/content/drive/MyDrive/models/NLP_Conv1D.keras",  # or use .h5
    save_best_only=True,
    monitor="val_loss",  # or "val_accuracy"
    mode="min",          # use "max" if monitoring accuracy
    save_weights_only=False
)

history_5 = model_5.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences,val_labels),
                        callbacks=[create_tensorboard_callback(SAVE_DIR,"Conv1D"),
                                   checkpoint_cb])

Saving TensorBoard log files to: /content/drive/MyDrive/models/model_logs/Conv1D/20250619-173730
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.6580 - loss: 0.6281 - val_accuracy: 0.7769 - val_loss: 0.4740
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.8454 - loss: 0.3709 - val_accuracy: 0.7835 - val_loss: 0.4767
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.9139 - loss: 0.2332 - val_accuracy: 0.7756 - val_loss: 0.5329
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.9508 - loss: 0.1482 - val_accuracy: 0.7664 - val_loss: 0.6042
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.9681 - loss: 0.1005 - val_accuracy: 0.7612 - val_loss: 0.6680


In [ ]:
model_5_pred = model_5.predict(val_sentences)
model_5_preds = tf.squeeze(tf.round(model_5_pred))
model_5_preds[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([1., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [ ]:
model_5_results = calculate_results(y_true=val_labels,
                                    y_pred=model_5_preds)
model_5_results

{'accuracy': 76.11548556430446,
 'precision': 0.7625228137873495,
 'recall': 0.7611548556430446,
 'f1': 0.7589888106092002}

In [ ]:
compare_baseline_to_new_results(loaded_baseline_results, model_5_results)

Baseline accuracy: 79.27, New accuracy: 76.12, Difference: -3.15
Baseline precision: 0.81, New precision: 0.76, Difference: -0.05
Baseline recall: 0.79, New recall: 0.76, Difference: -0.03
Baseline f1: 0.79, New f1: 0.76, Difference: -0.03
